### Cell code 01: Kết nối Drive và nạp thư viện

- **Mục đích:** Nạp các thư viện phân tích và kết nối Google Drive trong Colab.
- **Vì sao:** EDA cần truy cập bảng Olist đã xử lý và dùng chung đường dẫn dữ liệu.
- **Giúp ích:** Có môi trường sẵn sàng để chạy các cell tiếp theo.


In [1]:

from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
print("Đã kết nối Google Drive:", DRIVE_ROOT.exists())

Mounted at /content/drive
Đã kết nối Google Drive: True


### Cell code 02: Xác định thư mục dữ liệu đã xử lý

- **Mục đích:** Khai báo đường dẫn data/processed và liệt kê các tệp CSV hiện có.
- **Vì sao:** Cần xác nhận đầu vào EDA đã được tạo ở bước tiền xử lý.
- **Giúp ích:** Phát hiện sớm thiếu tệp hoặc dùng nhầm thư mục.


In [2]:

from pathlib import Path

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")
processed_dir = data_dir / "processed"

print("Thư mục dữ liệu:", data_dir)
print("Nội dung processed:")
for p in sorted(processed_dir.iterdir()):
    print(" -", p.name)


Thư mục dữ liệu: /content/drive/MyDrive/BigData/DA-BigData/code/data
Nội dung processed:
 - customer_churn_features_final
 - customer_churn_model_dataset
 - olist_churn_smotenc_ready
 - olist_customer_churn_features_selected
 - olist_customer_churn_features_temporal
 - olist_customer_churn_snapshots_90_120_180
 - olist_customer_churn_snapshots_eligible_180
 - olist_customer_purchase_event_snapshots
 - olist_customer_purchase_event_snapshots_labeled_180
 - olist_eda_delivered_orders
 - olist_eda_delivered_orders.csv
 - olist_order_payments_prepared.csv
 - olist_orders_prepared.csv
 - olist_products_prepared.csv
 - olist_review_order_links_audit.csv
 - olist_reviews_per_customer.csv


### Cell code 03: Khởi tạo PySpark và đọc bảng nền EDA từ Processing


In [5]:
%pip -q install pyspark

from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("OlistEDA")
    .getOrCreate()
)

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")
processed_dir = data_dir / "processed"

def read_processed_csv(path, multiline=False):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy dữ liệu đã xử lý: {path}\n"
            "Hãy chạy 02_Data_Preparation_and_Preprocessing.ipynb trước."
        )

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", multiline)
        .csv(str(path))
    )

eda_folder = processed_dir / "olist_eda_delivered_orders"
eda_csv = processed_dir / "olist_eda_delivered_orders.csv"

if eda_folder.exists():
    eda_path = eda_folder
elif eda_csv.exists():
    eda_path = eda_csv
else:
    raise FileNotFoundError(
        "Không tìm thấy olist_eda_delivered_orders trong thư mục processed."
    )

orders_prepared_path = processed_dir / "olist_orders_prepared.csv"
payments_prepared_path = processed_dir / "olist_order_payments_prepared.csv"
products_prepared_path = processed_dir / "olist_products_prepared.csv"
review_links_path = processed_dir / "olist_review_order_links_audit.csv"
reviews_customer_path = processed_dir / "olist_reviews_per_customer.csv"

eda_orders = read_processed_csv(eda_path)

orders_prepared = read_processed_csv(
    orders_prepared_path
)

order_payments_prepared = read_processed_csv(
    payments_prepared_path
)

products_prepared = read_processed_csv(
    products_prepared_path
)

review_links_processed = read_processed_csv(
    review_links_path,
    multiline=True
)

reviews_per_customer = read_processed_csv(
    reviews_customer_path,
    multiline=True
)

timestamp_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for c in timestamp_columns:
    if c in orders_prepared.columns:
        orders_prepared = orders_prepared.withColumn(
            c,
            F.to_timestamp(F.col(c))
        )

eda_orders = eda_orders.withColumn(
    "purchase_at",
    F.to_timestamp("order_purchase_timestamp")
)

eda_casts = {
    "item_count": "int",
    "payment_count": "int",
    "total_price": "double",
    "total_freight": "double",
    "total_items_and_freight": "double",
    "total_payment": "double",
    "payment_difference": "double",
    "absolute_difference": "double",
    "product_item_rows": "int",
    "missing_product_rows": "int",
    "missing_category_item_count": "int",
    "has_missing_product_category": "int"
}

for c, dtype in eda_casts.items():
    if c in eda_orders.columns:
        eda_orders = eda_orders.withColumn(
            c,
            F.col(c).cast(dtype)
        )

orders_casts = {
    "co_the_tinh_dac_trung_giao_hang": "int",
    "so_ngay_tu_dat_den_giao": "double",
    "so_ngay_tu_ban_giao_den_giao": "double",
    "giao_tre_so_voi_du_kien": "int",
    "co_ban_giao_truoc_dat": "int",
    "co_ban_giao_truoc_duyet": "int",
    "co_giao_truoc_ban_giao": "int",
    "co_thieu_ngay_giao_delivered": "int",
    "co_thieu_ngay_ban_giao_delivered": "int",
    "co_thieu_ngay_duyet_delivered": "int"
}

for c, dtype in orders_casts.items():
    if c in orders_prepared.columns:
        orders_prepared = orders_prepared.withColumn(
            c,
            F.col(c).cast(dtype)
        )

required_eda_columns = [
    "order_id",
    "customer_unique_id",
    "order_purchase_timestamp",
    "item_count",
    "total_freight",
    "total_payment",
    "reconciliation_status"
]

missing_eda_columns = [
    c for c in required_eda_columns
    if c not in eda_orders.columns
]

if missing_eda_columns:
    raise ValueError(
        "Bảng EDA thiếu các cột cần thiết: "
        + ", ".join(missing_eda_columns)
    )

eda_rows = eda_orders.count()
eda_unique_orders = (
    eda_orders
    .select("order_id")
    .distinct()
    .count()
)

orders_rows = orders_prepared.count()
orders_unique = (
    orders_prepared
    .select("order_id")
    .distinct()
    .count()
)

payment_rows = order_payments_prepared.count()
payment_unique = (
    order_payments_prepared
    .select("order_id")
    .distinct()
    .count()
)

product_rows = products_prepared.count()
product_unique = (
    products_prepared
    .select("product_id")
    .distinct()
    .count()
)

if eda_rows != eda_unique_orders:
    raise ValueError(
        "olist_eda_delivered_orders không đạt grain 1 dòng = 1 order."
    )

if orders_rows != orders_unique:
    raise ValueError(
        "olist_orders_prepared có order_id bị trùng."
    )

if payment_rows != payment_unique:
    raise ValueError(
        "olist_order_payments_prepared không đạt grain 1 dòng = 1 order."
    )

if product_rows != product_unique:
    raise ValueError(
        "olist_products_prepared không đạt grain 1 dòng = 1 product."
    )

print("=" * 70)
print("DỮ LIỆU PROCESSED ĐÃ NẠP CHO EDA")
print("=" * 70)

print(f"EDA delivered orders      : {eda_rows:,}")
print(f"Order ID duy nhất         : {eda_unique_orders:,}")
print(
    f"Customer duy nhất         : "
    f"{eda_orders.select('customer_unique_id').distinct().count():,}"
)
print(f"Orders prepared           : {orders_rows:,}")
print(f"Payments prepared         : {payment_rows:,}")
print(f"Products prepared         : {product_rows:,}")
print(
    f"Review links processed    : "
    f"{review_links_processed.count():,}"
)
print(
    f"Reviews/customer processed: "
    f"{reviews_per_customer.count():,}"
)

print("\n✓ Chỉ sử dụng dữ liệu đã xử lý từ processed/")
print("✓ Grain bảng nền EDA: 1 dòng = 1 delivered order")

DỮ LIỆU PROCESSED ĐÃ NẠP CHO EDA
EDA delivered orders      : 96,478
Order ID duy nhất         : 96,478
Customer duy nhất         : 93,358
Orders prepared           : 99,441
Payments prepared         : 99,441
Products prepared         : 32,951
Review links processed    : 99,224
Reviews/customer processed: 98,410

✓ Chỉ sử dụng dữ liệu đã xử lý từ processed/
✓ Grain bảng nền EDA: 1 dòng = 1 delivered order


### Cell code 04: Tổng quan hành vi mua và mua lại của khách hàng


In [6]:
%pip -q install hvplot panel jupyter_bokeh

import pandas as pd
import hvplot.pandas
import panel as pn

from pyspark.sql import functions as F

pn.extension()

eda = eda_orders

delivered_ids = (
    eda
    .select("order_id")
    .dropDuplicates()
)

order_values = (
    eda
    .select(
        "order_id",
        "customer_unique_id",
        "item_count",
        "payment_count",
        "total_price",
        "total_freight",
        "total_payment",
        "reconciliation_status",
        "payment_matched_flag",
        "payment_mismatch_flag",
        "payment_missing_flag",
        "has_missing_product_category",
        "missing_category_item_count"
    )
)

money_valid = (
    order_values
    .filter(
        F.col("reconciliation_status") == "KHOP"
    )
)

delivery = (
    orders_prepared
    .join(
        delivered_ids,
        "order_id",
        "inner"
    )
    .withColumn(
        "delivery_days",
        F.col("so_ngay_tu_dat_den_giao").cast("double")
    )
    .withColumn(
        "late_status",
        F.when(
            F.col("co_the_tinh_dac_trung_giao_hang") != 1,
            "Thiếu/sai mốc giao"
        )
        .when(
            F.col("order_estimated_delivery_date").isNull(),
            "Thiếu ngày dự kiến"
        )
        .when(
            F.col("giao_tre_so_voi_du_kien") == 1,
            "Giao trễ"
        )
        .when(
            F.col("giao_tre_so_voi_du_kien") == 0,
            "Đúng hạn hoặc sớm"
        )
        .otherwise(
            "Không xác định"
        )
    )
)

review_ids = (
    review_links_processed
    .filter(
        F.col("review_id").isNotNull()
    )
    .groupBy("review_id")
    .agg(
        F.countDistinct("order_id").alias(
            "linked_orders"
        ),
        F.countDistinct(
            F.col("review_score").cast("int")
        ).alias(
            "different_scores"
        ),
        F.min(
            F.col("review_score").cast("int")
        ).alias(
            "score"
        )
    )
    .withColumn(
        "score_group",
        F.when(
            F.col("different_scores") > 1,
            "Điểm không nhất quán"
        )
        .when(
            F.col("score").between(1, 5),
            F.col("score").cast("string")
        )
        .otherwise(
            "Thiếu/không hợp lệ"
        )
    )
)

def numeric_summary(
    df,
    column,
    label
):
    row = (
        df
        .agg(
            F.count("*").alias("rows"),
            F.count(column).alias("available"),
            F.min(column).alias("minimum"),
            F.percentile_approx(
                column,
                [0.5, 0.75, 0.9, 0.95, 0.99],
                10000
            ).alias("quantiles"),
            F.max(column).alias("maximum")
        )
        .first()
    )

    quantiles = (
        row["quantiles"]
        if row["quantiles"] is not None
        else [None] * 5
    )

    return {
        "Chỉ tiêu": label,
        "Số dòng": row["rows"],
        "Có giá trị": row["available"],
        "Thiếu": (
            row["rows"]
            - row["available"]
        ),
        "Nhỏ nhất": row["minimum"],
        "P50": quantiles[0],
        "P75": quantiles[1],
        "P90": quantiles[2],
        "P95": quantiles[3],
        "P99": quantiles[4],
        "Lớn nhất": row["maximum"]
    }

summary = pd.DataFrame([
    numeric_summary(
        money_valid,
        "total_payment",
        "Tiền thanh toán/đơn khớp"
    ),
    numeric_summary(
        money_valid,
        "total_freight",
        "Phí vận chuyển/đơn khớp"
    ),
    numeric_summary(
        order_values,
        "item_count",
        "Số item/đơn delivered"
    ),
    numeric_summary(
        delivery,
        "delivery_days",
        "Số ngày từ đặt đến giao hợp lệ"
    )
]).round(2)

def bucket_plot(
    df,
    column,
    cuts,
    labels,
    title,
    width=500
):
    expression = None

    for boundary, label in zip(
        cuts,
        labels
    ):
        condition = (
            F.col(column)
            < boundary
        )

        expression = (
            F.when(
                condition,
                label
            )
            if expression is None
            else expression.when(
                condition,
                label
            )
        )

    expression = expression.otherwise(
        labels[-1]
    )

    grouped = (
        df
        .filter(
            F.col(column).isNotNull()
        )
        .withColumn(
            "khoang",
            expression
        )
        .groupBy("khoang")
        .count()
        .toPandas()
    )

    grouped["thu_tu"] = (
        grouped["khoang"]
        .map({
            label: i
            for i, label
            in enumerate(labels)
        })
    )

    grouped = grouped.sort_values(
        "thu_tu"
    )

    return grouped.hvplot.bar(
        x="khoang",
        y="count",
        width=width,
        height=330,
        rot=45,
        title=title,
        ylabel="Số lượng",
        xlabel=""
    )

payment_chart = bucket_plot(
    money_valid,
    "total_payment",
    [50, 100, 200, 500, 1000],
    [
        "<50",
        "50–<100",
        "100–<200",
        "200–<500",
        "500–<1000",
        "≥1000"
    ],
    "Tiền thanh toán theo đơn khớp"
)

freight_chart = bucket_plot(
    money_valid,
    "total_freight",
    [1, 10, 20, 40, 80],
    [
        "<1",
        "1–<10",
        "10–<20",
        "20–<40",
        "40–<80",
        "≥80"
    ],
    "Phí vận chuyển theo đơn khớp"
)

item_chart = bucket_plot(
    order_values,
    "item_count",
    [2, 3, 4, 6],
    [
        "1",
        "2",
        "3",
        "4–5",
        "≥6"
    ],
    "Số item trên mỗi đơn"
)

delivery_chart = bucket_plot(
    delivery,
    "delivery_days",
    [3, 7, 14, 30, 60],
    [
        "<3",
        "3–<7",
        "7–<14",
        "14–<30",
        "30–<60",
        "≥60"
    ],
    "Số ngày từ đặt đến giao hợp lệ"
)

review_counts = (
    review_ids
    .groupBy("score_group")
    .count()
    .toPandas()
)

review_counts["thu_tu"] = (
    review_counts["score_group"]
    .map({
        "1": 1,
        "2": 2,
        "3": 3,
        "4": 4,
        "5": 5,
        "Điểm không nhất quán": 6,
        "Thiếu/không hợp lệ": 7
    })
)

review_counts = review_counts.sort_values(
    "thu_tu"
)

review_chart = review_counts.hvplot.bar(
    x="score_group",
    y="count",
    width=500,
    height=330,
    title="Điểm review từ dữ liệu review đã xử lý",
    ylabel="Số review_id",
    xlabel=""
)

late_counts = (
    delivery
    .groupBy("late_status")
    .count()
    .toPandas()
)

late_chart = late_counts.hvplot.bar(
    x="late_status",
    y="count",
    width=500,
    height=330,
    rot=25,
    title="Trạng thái giao hàng từ orders_prepared",
    ylabel="Số đơn",
    xlabel=""
)

product_categories = (
    products_prepared
    .withColumn(
        "category",
        F.when(
            F.col(
                "product_category_name"
            ).isNull()
            |
            (
                F.trim(
                    F.col(
                        "product_category_name"
                    )
                ) == ""
            ),
            "Thiếu danh mục"
        )
        .otherwise(
            F.col(
                "product_category_name"
            )
        )
    )
)

category_counts = (
    product_categories
    .groupBy("category")
    .agg(
        F.count("*").alias(
            "so_san_pham"
        )
    )
    .orderBy(
        F.desc("so_san_pham")
    )
    .toPandas()
)

top_categories = (
    category_counts
    .head(15)
    .copy()
)

other_count = int(
    category_counts
    .iloc[15:]["so_san_pham"]
    .sum()
)

if other_count > 0:
    top_categories = pd.concat([
        top_categories,
        pd.DataFrame([{
            "category":
                "Các danh mục còn lại",
            "so_san_pham":
                other_count
        }])
    ], ignore_index=True)

category_chart = (
    top_categories
    .hvplot.barh(
        x="category",
        y="so_san_pham",
        width=650,
        height=460,
        title=(
            "Phân bố danh mục trong "
            "products_prepared"
        ),
        xlabel="Số sản phẩm",
        ylabel=""
    )
)

payment_count_df = (
    order_values
    .filter(
        F.col(
            "payment_count"
        ).isNotNull()
    )
    .withColumn(
        "payment_count_group",
        F.when(
            F.col("payment_count") <= 1,
            "1"
        )
        .when(
            F.col("payment_count") == 2,
            "2"
        )
        .when(
            F.col("payment_count") == 3,
            "3"
        )
        .when(
            F.col("payment_count") == 4,
            "4"
        )
        .otherwise("≥5")
    )
    .groupBy(
        "payment_count_group"
    )
    .count()
    .toPandas()
)

payment_count_df["thu_tu"] = (
    payment_count_df[
        "payment_count_group"
    ]
    .map({
        "1": 1,
        "2": 2,
        "3": 3,
        "4": 4,
        "≥5": 5
    })
)

payment_count_df = (
    payment_count_df
    .sort_values("thu_tu")
)

payment_count_chart = (
    payment_count_df
    .hvplot.bar(
        x="payment_count_group",
        y="count",
        width=500,
        height=330,
        title=(
            "Số dòng thanh toán "
            "trên mỗi đơn"
        ),
        xlabel="Số dòng thanh toán",
        ylabel="Số đơn"
    )
)

reconciliation_counts = (
    order_values
    .groupBy(
        "reconciliation_status"
    )
    .count()
    .orderBy(
        F.desc("count")
    )
    .toPandas()
)

reconciliation_chart = (
    reconciliation_counts
    .hvplot.bar(
        x="reconciliation_status",
        y="count",
        width=500,
        height=330,
        rot=25,
        title=(
            "Trạng thái đối chiếu "
            "tiền theo đơn"
        ),
        xlabel="",
        ylabel="Số đơn"
    )
)

review_multi_order = (
    review_ids
    .filter(
        F.col("linked_orders") > 1
    )
    .count()
)

review_inconsistent = (
    review_ids
    .filter(
        F.col(
            "different_scores"
        ) > 1
    )
    .count()
)

products_missing_category = (
    product_categories
    .filter(
        F.col("category")
        == "Thiếu danh mục"
    )
    .count()
)

orders_missing_category = (
    order_values
    .filter(
        F.col(
            "has_missing_product_category"
        ) == 1
    )
    .count()
)

orders_missing_payment = (
    order_values
    .filter(
        F.col("payment_missing_flag")
        .cast("int") == 1
    )
    .count()
)

orders_payment_mismatch = (
    order_values
    .filter(
        F.col("payment_mismatch_flag")
        .cast("int") == 1
    )
    .count()
)

quality = pd.DataFrame([
    {
        "Kiểm tra":
            "Review ID gắn với nhiều order_id",
        "Kết quả":
            review_multi_order
    },
    {
        "Kiểm tra":
            "Review ID có điểm không nhất quán",
        "Kết quả":
            review_inconsistent
    },
    {
        "Kiểm tra":
            "Sản phẩm thiếu danh mục",
        "Kết quả":
            products_missing_category
    },
    {
        "Kiểm tra":
            "Đơn có sản phẩm thiếu danh mục",
        "Kết quả":
            orders_missing_category
    },
    {
        "Kiểm tra":
            "Đơn thiếu payment",
        "Kết quả":
            orders_missing_payment
    },
    {
        "Kiểm tra":
            "Đơn lệch tiền",
        "Kết quả":
            orders_payment_mismatch
    }
])

print(
    "PHÂN VỊ TRÊN TOÀN BỘ GIÁ TRỊ HỢP LỆ"
)
display(summary)

print(
    "CÁC TRƯỜNG HỢP CHẤT LƯỢNG "
    "CẦN TÁCH RIÊNG"
)
display(quality)

print(
    "TRẠNG THÁI ĐỐI CHIẾU TIỀN "
    "TRÊN DELIVERED ORDERS"
)
display(reconciliation_counts)

display(
    pn.Column(
        pn.Row(
            payment_chart,
            freight_chart
        ),
        pn.Row(
            item_chart,
            delivery_chart
        ),
        pn.Row(
            review_chart,
            late_chart
        ),
        pn.Row(
            category_chart,
            payment_count_chart
        ),
        reconciliation_chart
    )
)

PHÂN VỊ TRÊN TOÀN BỘ GIÁ TRỊ HỢP LỆ


,Chỉ tiêu,Số dòng,Có giá trị,Thiếu,Nhỏ nhất,P50,P75,P90,P95,P99,Lớn nhất
0,Tiền thanh toán/đơn khớp,95908,95908,0,9.59,105.07,175.82,304.27,442.63,1043.51,13664.08
1,Phí vận chuyển/đơn khớp,95908,95908,0,0.00,17.16,23.92,39.06,54.24,102.55,1794.96
2,Số item/đơn delivered,96478,96478,0,1.00,1.00,1.00,1.00,2.00,3.00,21.00
3,Số ngày từ đặt đến giao hợp lệ,96478,95082,1396,0.53,10.27,15.81,23.15,29.38,46.05,209.63


CÁC TRƯỜNG HỢP CHẤT LƯỢNG CẦN TÁCH RIÊNG


,Kiểm tra,Kết quả
0,Review ID gắn với nhiều order_id,789
1,Review ID có điểm không nhất quán,0
2,Sản phẩm thiếu danh mục,610
3,Đơn có sản phẩm thiếu danh mục,1392
4,Đơn thiếu payment,1
5,Đơn lệch tiền,569


TRẠNG THÁI ĐỐI CHIẾU TIỀN TRÊN DELIVERED ORDERS


,reconciliation_status,count
0,KHOP,95908
1,LECH,569
2,THIEU_THANH_TOAN,1


Column
    [0] Row
        [0] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [1] Row
        [0] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [2] Row
        [0] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [3] Row
        [0] HoloViews(Bars, height=460, sizing_mode='fixed', width=650)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [4] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)

### Cell code 05: Cài và nạp thư viện trực quan hóa

- **Mục đích:** Chuẩn bị hvPlot và Panel để hiển thị biểu đồ, bảng thống kê.
- **Vì sao:** Biểu đồ giúp quan sát kết quả tổng hợp dễ hơn bảng số đơn thuần.
- **Giúp ích:** Các cell phân tích sau có thể trình bày kết quả trực quan.


In [7]:
%pip -q install hvplot panel

from pathlib import Path
import pandas as pd
import hvplot.pandas
import panel as pn

### Cell code 06: Phân tích cohort theo tháng mua đầu

- **Mục đích:** Nhóm khách theo tháng mua đầu và tính quy mô, tỷ lệ mua lần hai ở các cohort đủ 180 ngày theo dõi.
- **Vì sao:** Cohort gần cuối dữ liệu chưa có cơ hội mua lại đủ lâu để so sánh công bằng.
- **Giúp ích:** Thấy khác biệt mua lại giữa các nhóm khách bắt đầu ở thời điểm khác nhau.


In [8]:
from pathlib import Path
import pandas as pd
import hvplot.pandas
import panel as pn
from pyspark.sql import functions as F
from pyspark.sql.window import Window

pn.extension()

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")

eda_delivered = (
    eda_orders.select("order_id", "customer_unique_id", "order_purchase_timestamp")
    .withColumn("purchase_at", F.to_timestamp("order_purchase_timestamp"))
    .filter(F.col("customer_unique_id").isNotNull() & F.col("purchase_at").isNotNull())
)

data_end = eda_delivered.agg(
    F.max("purchase_at").alias("data_end")
).first()["data_end"]

purchase_window = Window.partitionBy(
    "customer_unique_id"
).orderBy(
    "purchase_at", "order_id"
)

ranked_purchases = (
    eda_delivered
    .withColumn("purchase_rank", F.row_number().over(purchase_window))
    .filter(F.col("purchase_rank") <= 2)
)

customer_purchase_pairs = (
    ranked_purchases
    .groupBy("customer_unique_id")
    .agg(
        F.max(
            F.when(F.col("purchase_rank") == 1, F.col("purchase_at"))
        ).alias("first_purchase_at"),
        F.max(
            F.when(F.col("purchase_rank") == 2, F.col("purchase_at"))
        ).alias("second_purchase_at")
    )
    .withColumn(
        "cohort_month",
        F.trunc(F.to_date("first_purchase_at"), "month")
    )
    .withColumn(
        "full_180_day_followup",
        F.when(
            F.date_add(F.last_day("cohort_month"), 180)
            < F.lit(data_end.date()),
            1
        ).otherwise(0)
    )
)

cohort_coverage = (
    customer_purchase_pairs
    .groupBy("cohort_month", "full_180_day_followup")
    .agg(F.count("*").alias("so_khach"))
    .orderBy("cohort_month")
)

eligible_customers = customer_purchase_pairs.filter(
    F.col("full_180_day_followup") == 1
)

repeat_counts = []

for days in [30, 60, 90, 120, 180]:
    repeat_counts.append(
        F.sum(
            F.when(
                F.col("second_purchase_at").isNotNull() &
                (
                    F.col("second_purchase_at")
                    <= F.expr(f"first_purchase_at + INTERVAL {days} DAYS")
                ),
                1
            ).otherwise(0)
        ).alias(f"repeat_{days}")
    )

cohort_summary = (
    eligible_customers
    .groupBy("cohort_month")
    .agg(
        F.count("*").alias("customers"),
        *repeat_counts
    )
)

for days in [30, 60, 90, 120, 180]:
    cohort_summary = cohort_summary.withColumn(
        f"repeat_{days}_pct",
        F.round(
            F.col(f"repeat_{days}") * 100 / F.col("customers"),
            2
        )
    )

cohort_summary = cohort_summary.orderBy("cohort_month")

print(f"Ngày mua delivered cuối dữ liệu: {data_end}")
print(f"Tổng số khách: {customer_purchase_pairs.count()}")
print(f"Số khách thuộc các cohort đủ 180 ngày theo dõi: {eligible_customers.count()}")

print("COHORT NÀO ĐỦ ĐIỀU KIỆN SO SÁNH")
cohort_coverage.show(100, truncate=False)

print("TỶ LỆ MUA LẦN HAI THEO THÁNG MUA ĐẦU")
cohort_summary.show(100, truncate=False)

cohort_chart = cohort_summary.toPandas()
cohort_chart["cohort_month"] = pd.to_datetime(
    cohort_chart["cohort_month"]
)

rates_chart = cohort_chart.hvplot.line(
    x="cohort_month",
    y=[
        "repeat_30_pct",
        "repeat_60_pct",
        "repeat_90_pct",
        "repeat_120_pct",
        "repeat_180_pct"
    ],
    width=1250,
    height=440,
    line_width=2,
    marker="o",
    ylabel="Tỷ lệ mua lần hai (%)",
    xlabel="Tháng mua đầu tiên",
    title="Tỷ lệ mua lần hai theo cohort"
)

sizes_chart = cohort_chart.hvplot.bar(
    x="cohort_month",
    y="customers",
    width=1250,
    height=380,
    rot=45,
    color="#3274A1",
    ylabel="Số khách",
    xlabel="Tháng mua đầu tiên",
    title="Quy mô các cohort đủ 180 ngày theo dõi"
)

display(pn.Column(rates_chart, sizes_chart))


Ngày mua delivered cuối dữ liệu: 2018-08-29 15:00:37
Tổng số khách: 93358
Số khách thuộc các cohort đủ 180 ngày theo dõi: 55525
COHORT NÀO ĐỦ ĐIỀU KIỆN SO SÁNH
+------------+---------------------+--------+
|cohort_month|full_180_day_followup|so_khach|
+------------+---------------------+--------+
|2016-09-01  |1                    |1       |
|2016-10-01  |1                    |262     |
|2016-12-01  |1                    |1       |
|2017-01-01  |1                    |717     |
|2017-02-01  |1                    |1628    |
|2017-03-01  |1                    |2503    |
|2017-04-01  |1                    |2256    |
|2017-05-01  |1                    |3451    |
|2017-06-01  |1                    |3037    |
|2017-07-01  |1                    |3752    |
|2017-08-01  |1                    |4057    |
|2017-09-01  |1                    |4004    |
|2017-10-01  |1                    |4328    |
|2017-11-01  |1                    |7060    |
|2017-12-01  |1                    |5338    |
|2018-01-01 

Column
    [0] HoloViews(NdOverlay, height=440, sizing_mode='fixed', width=1250)
    [1] HoloViews(Bars, height=380, sizing_mode='fixed', width=1250)

### Cell code 07: Phân tích khoảng cách giữa các lần mua

- **Mục đích:** Tính khoảng cách lần một–lần hai và các lần liên tiếp, gồm phân vị 50/75/90/95 và phân bố theo khoảng.
- **Vì sao:** Giá trị trung bình không thể hiện đầy đủ khách quay lại sớm hay muộn.
- **Giúp ích:** Có căn cứ quan sát để so sánh các ngưỡng churn 90, 120 và 180 ngày.


In [9]:
from pathlib import Path
import pandas as pd
import hvplot.pandas
import panel as pn
from pyspark.sql import functions as F
from pyspark.sql.window import Window

pn.extension()

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")

purchases = (
    eda_orders.select("order_id", "customer_unique_id", "order_purchase_timestamp")
    .withColumn("purchase_at", F.to_timestamp("order_purchase_timestamp"))
    .filter(F.col("customer_unique_id").isNotNull() & F.col("purchase_at").isNotNull())
)

purchase_window = Window.partitionBy(
    "customer_unique_id"
).orderBy(
    "purchase_at", "order_id"
)

purchase_gaps = (
    purchases
    .withColumn(
        "purchase_number",
        F.row_number().over(purchase_window)
    )
    .withColumn(
        "previous_purchase_at",
        F.lag("purchase_at").over(purchase_window)
    )
    .filter(F.col("previous_purchase_at").isNotNull())
    .withColumn(
        "gap_days",
        (
            F.unix_timestamp("purchase_at") -
            F.unix_timestamp("previous_purchase_at")
        ) / F.lit(86400.0)
    )
    .withColumn(
        "same_calendar_day",
        F.when(
            F.to_date("purchase_at") ==
            F.to_date("previous_purchase_at"),
            1
        ).otherwise(0)
    )
)

first_to_second = (
    purchase_gaps
    .filter(F.col("purchase_number") == 2)
    .withColumn("scope", F.lit("Lần 1 đến lần 2"))
)

all_consecutive = (
    purchase_gaps
    .withColumn("scope", F.lit("Mọi cặp mua liên tiếp"))
)

gap_scopes = first_to_second.unionByName(all_consecutive)

gap_stats = (
    gap_scopes
    .groupBy("scope")
    .agg(
        F.count("*").alias("so_cap_mua"),
        F.countDistinct("customer_unique_id").alias("so_khach"),
        F.sum("same_calendar_day").alias("so_cap_cung_ngay"),
        F.round(F.min("gap_days"), 2).alias("ngan_nhat_ngay"),
        F.round(F.avg("gap_days"), 2).alias("trung_binh_ngay"),
        F.round(F.max("gap_days"), 2).alias("lau_nhat_ngay"),
        F.expr(
            "percentile_approx("
            "gap_days, array(0.5, 0.75, 0.9, 0.95), 100000"
            ")"
        ).alias("percentiles")
    )
    .select(
        "scope",
        "so_cap_mua",
        "so_khach",
        "so_cap_cung_ngay",
        "ngan_nhat_ngay",
        "trung_binh_ngay",
        "lau_nhat_ngay",
        F.round(F.col("percentiles")[0], 2).alias("p50_ngay"),
        F.round(F.col("percentiles")[1], 2).alias("p75_ngay"),
        F.round(F.col("percentiles")[2], 2).alias("p90_ngay"),
        F.round(F.col("percentiles")[3], 2).alias("p95_ngay")
    )
)

bucketed_gaps = gap_scopes.withColumn(
    "bucket_order",
    F.when(F.col("gap_days") < 1, 0)
    .when(F.col("gap_days") < 7, 1)
    .when(F.col("gap_days") < 30, 2)
    .when(F.col("gap_days") < 60, 3)
    .when(F.col("gap_days") < 90, 4)
    .when(F.col("gap_days") < 120, 5)
    .when(F.col("gap_days") < 180, 6)
    .when(F.col("gap_days") < 365, 7)
    .otherwise(8)
)

bucket_counts = (
    bucketed_gaps
    .groupBy("scope", "bucket_order")
    .agg(F.count("*").alias("so_cap_mua"))
    .orderBy("bucket_order", "scope")
)

labels = [
    "Dưới 1 ngày",
    "1–<7 ngày",
    "7–<30 ngày",
    "30–<60 ngày",
    "60–<90 ngày",
    "90–<120 ngày",
    "120–<180 ngày",
    "180–<365 ngày",
    "Từ 365 ngày"
]

chart_data = (
    bucket_counts
    .toPandas()
    .pivot(
        index="bucket_order",
        columns="scope",
        values="so_cap_mua"
    )
    .reindex(range(len(labels)), fill_value=0)
    .fillna(0)
    .reset_index(drop=True)
)

chart_data["Khoảng cách"] = labels

print("PHÂN VỊ KHOẢNG CÁCH GIỮA CÁC LẦN MUA")
gap_stats.show(truncate=False)

print("PHÂN BỐ SỐ CẶP MUA THEO KHOẢNG CÁCH")
bucket_counts.show(100, truncate=False)

gap_chart = chart_data.hvplot.bar(
    x="Khoảng cách",
    y=["Lần 1 đến lần 2", "Mọi cặp mua liên tiếp"],
    stacked=False,
    width=1350,
    height=460,
    rot=40,
    ylabel="Số cặp đơn",
    title="Phân bố khoảng cách giữa các lần mua"
)

display(pn.Column(gap_chart))


PHÂN VỊ KHOẢNG CÁCH GIỮA CÁC LẦN MUA
+---------------------+----------+--------+----------------+--------------+---------------+-------------+--------+--------+--------+--------+
|scope                |so_cap_mua|so_khach|so_cap_cung_ngay|ngan_nhat_ngay|trung_binh_ngay|lau_nhat_ngay|p50_ngay|p75_ngay|p90_ngay|p95_ngay|
+---------------------+----------+--------+----------------+--------------+---------------+-------------+--------+--------+--------+--------+
|Mọi cặp mua liên tiếp|3120      |2801    |890             |0.0           |79.15          |608.98       |29.42   |121.43  |240.21  |312.16  |
|Lần 1 đến lần 2      |2801      |2801    |829             |0.0           |81.21          |608.98       |28.98   |126.26  |246.31  |324.08  |
+---------------------+----------+--------+----------------+--------------+---------------+-------------+--------+--------+--------+--------+

PHÂN BỐ SỐ CẶP MUA THEO KHOẢNG CÁCH
+---------------------+------------+----------+
|scope                |buc

Column
    [0] HoloViews(Bars, height=460, sizing_mode='fixed', width=1350)

### Cell code 08: Khảo sát phân bố và chất lượng feature

- **Mục đích:** Xem tiền, freight, số item, review, giao hàng, danh mục, thanh toán và số kỳ trả góp; tách dữ liệu thiếu/đã đánh cờ.
- **Vì sao:** Các ngoại lệ và liên kết review mơ hồ có thể khiến biểu đồ bị hiểu sai.
- **Giúp ích:** Biết đặc trưng nào hữu ích và phạm vi dữ liệu hợp lệ của từng thống kê.


In [10]:
%pip -q install hvplot panel jupyter_bokeh

from pathlib import Path
import pandas as pd
import hvplot.pandas
import panel as pn
from pyspark.sql import functions as F

pn.extension()

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")
processed_dir = data_dir / "processed"

def read_csv(path, multiline=False):
    return (
        spark.read.option("header", True)
        .option("multiLine", multiline)
        .option("escape", '"')
        .csv(str(path))
    )

eda = eda_orders
orders = read_csv(data_dir / "olist_orders_dataset.csv")
items = read_csv(data_dir / "olist_order_items_dataset.csv")
payments = read_csv(data_dir / "olist_order_payments_dataset.csv")
products = read_csv(data_dir / "olist_products_dataset.csv")
reviews = read_csv(data_dir / "olist_order_reviews_dataset.csv", multiline=True)
payment_audit = eda.select("order_id", "reconciliation_status")

delivered_ids = eda.select("order_id").dropDuplicates()

item_orders = (
    items.join(delivered_ids, "order_id", "inner")
    .withColumn("price_num", F.col("price").cast("double"))
    .withColumn("freight_num", F.col("freight_value").cast("double"))
    .groupBy("order_id")
    .agg(
        F.count("*").alias("item_count"),
        F.sum("price_num").alias("total_price"),
        F.sum("freight_num").alias("total_freight")
    )
)

payment_orders = (
    payments.join(delivered_ids, "order_id", "inner")
    .withColumn("payment_num", F.col("payment_value").cast("double"))
    .groupBy("order_id")
    .agg(F.sum("payment_num").alias("total_payment"))
)

order_values = (
    delivered_ids
    .join(item_orders, "order_id", "left")
    .join(payment_orders, "order_id", "left")
    .join(
        payment_audit.select("order_id", "reconciliation_status"),
        "order_id",
        "left"
    )
)

money_valid = order_values.filter(F.col("reconciliation_status") == "KHOP")

delivery = (
    orders.join(delivered_ids, "order_id", "inner")
    .withColumn("purchased_at", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("carrier_at", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("delivered_at", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("estimated_on", F.to_date("order_estimated_delivery_date"))
    .withColumn(
        "valid_delivery",
        F.col("purchased_at").isNotNull()
        & F.col("carrier_at").isNotNull()
        & F.col("delivered_at").isNotNull()
        & (F.col("carrier_at") >= F.col("purchased_at"))
        & (F.col("delivered_at") >= F.col("carrier_at"))
    )
    .withColumn(
        "delivery_days",
        F.when(
            F.col("valid_delivery"),
            (F.unix_timestamp("delivered_at") - F.unix_timestamp("purchased_at")) / 86400
        )
    )
    .withColumn(
        "late_status",
        F.when(~F.col("valid_delivery"), "Thiếu/sai mốc giao")
        .when(F.col("estimated_on").isNull(), "Thiếu ngày dự kiến")
        .when(F.to_date("delivered_at") > F.col("estimated_on"), "Giao trễ")
        .otherwise("Đúng hạn hoặc sớm")
    )
)

review_ids = (
    reviews.filter(F.col("review_id").isNotNull())
    .groupBy("review_id")
    .agg(
        F.countDistinct("order_id").alias("linked_orders"),
        F.countDistinct("review_score").alias("different_scores"),
        F.min(F.col("review_score").cast("int")).alias("score")
    )
    .withColumn(
        "score_group",
        F.when(F.col("different_scores") > 1, "Điểm không nhất quán")
        .when(F.col("score").between(1, 5), F.col("score").cast("string"))
        .otherwise("Thiếu/không hợp lệ")
    )
)

def numeric_summary(df, column, label):
    row = df.agg(
        F.count("*").alias("rows"),
        F.count(column).alias("available"),
        F.min(column).alias("minimum"),
        F.percentile_approx(column, [0.5, 0.75, 0.9, 0.95, 0.99], 10000).alias("quantiles"),
        F.max(column).alias("maximum")
    ).first()
    quantiles = row["quantiles"] or [None] * 5
    return {
        "Chỉ tiêu": label,
        "Số dòng": row["rows"],
        "Có giá trị": row["available"],
        "Thiếu": row["rows"] - row["available"],
        "Nhỏ nhất": row["minimum"],
        "P50": quantiles[0],
        "P75": quantiles[1],
        "P90": quantiles[2],
        "P95": quantiles[3],
        "P99": quantiles[4],
        "Lớn nhất": row["maximum"]
    }

summary = pd.DataFrame([
    numeric_summary(money_valid, "total_payment", "Tiền thanh toán/đơn khớp"),
    numeric_summary(money_valid, "total_freight", "Phí vận chuyển/đơn khớp"),
    numeric_summary(order_values, "item_count", "Số item/đơn delivered"),
    numeric_summary(delivery, "delivery_days", "Số ngày từ đặt đến giao hợp lệ")
]).round(2)

def bucket_plot(df, column, cuts, labels, title, width=500):
    expression = None
    for boundary, label in zip(cuts, labels):
        condition = F.col(column) < boundary
        expression = F.when(condition, label) if expression is None else expression.when(condition, label)
    expression = expression.otherwise(labels[-1])
    grouped = (
        df.filter(F.col(column).isNotNull())
        .withColumn("khoang", expression)
        .groupBy("khoang")
        .count()
        .toPandas()
    )
    grouped["thu_tu"] = grouped["khoang"].map({label: i for i, label in enumerate(labels)})
    grouped = grouped.sort_values("thu_tu")
    return grouped.hvplot.bar(
        x="khoang", y="count", width=width, height=330,
        rot=45, title=title, ylabel="Số lượng", xlabel=""
    )

payment_chart = bucket_plot(
    money_valid, "total_payment",
    [50, 100, 200, 500, 1000],
    ["<50", "50–<100", "100–<200", "200–<500", "500–<1000", "≥1000"],
    "Tiền thanh toán theo đơn khớp"
)

freight_chart = bucket_plot(
    money_valid, "total_freight",
    [1, 10, 20, 40, 80],
    ["<1", "1–<10", "10–<20", "20–<40", "40–<80", "≥80"],
    "Phí vận chuyển theo đơn khớp"
)

item_chart = bucket_plot(
    order_values, "item_count",
    [2, 3, 4, 6],
    ["1", "2", "3", "4–5", "≥6"],
    "Số item trên mỗi đơn"
)

delivery_chart = bucket_plot(
    delivery, "delivery_days",
    [3, 7, 14, 30, 60],
    ["<3", "3–<7", "7–<14", "14–<30", "30–<60", "≥60"],
    "Số ngày từ đặt đến giao hợp lệ"
)

review_counts = review_ids.groupBy("score_group").count().toPandas()
review_counts["thu_tu"] = review_counts["score_group"].map(
    {"1": 1, "2": 2, "3": 3, "4": 4, "5": 5,
     "Điểm không nhất quán": 6, "Thiếu/không hợp lệ": 7}
)
review_counts = review_counts.sort_values("thu_tu")
review_chart = review_counts.hvplot.bar(
    x="score_group", y="count", width=500, height=330,
    title="Điểm review theo review_id duy nhất",
    ylabel="Số review_id", xlabel=""
)

late_counts = delivery.groupBy("late_status").count().toPandas()
late_chart = late_counts.hvplot.bar(
    x="late_status", y="count", width=500, height=330,
    rot=25, title="Giao trễ và các mốc không đánh giá được",
    ylabel="Số đơn", xlabel=""
)

item_categories = (
    items.join(delivered_ids, "order_id", "inner")
    .join(
        products.select("product_id", "product_category_name"),
        "product_id",
        "left"
    )
    .withColumn(
        "category",
        F.when(
            F.col("product_category_name").isNull()
            | (F.trim("product_category_name") == ""),
            "Thiếu danh mục"
        ).otherwise(F.col("product_category_name"))
    )
)

category_counts = (
    item_categories.groupBy("category")
    .agg(F.count("*").alias("so_item"))
    .orderBy(F.desc("so_item"))
    .toPandas()
)

top_categories = category_counts.head(15).copy()
other_count = int(category_counts.iloc[15:]["so_item"].sum())
if other_count:
    top_categories = pd.concat([
        top_categories,
        pd.DataFrame([{"category": "Các danh mục còn lại", "so_item": other_count}])
    ], ignore_index=True)

category_chart = top_categories.hvplot.barh(
    x="category", y="so_item", width=650, height=460,
    title="Nhóm sản phẩm theo số item đã bán",
    xlabel="Số item", ylabel=""
)

payment_lines = payments.join(delivered_ids, "order_id", "inner")

payment_types = (
    payment_lines.groupBy("payment_type")
    .agg(
        F.count("*").alias("so_dong_thanh_toan"),
        F.countDistinct("order_id").alias("so_don")
    )
    .orderBy(F.desc("so_don"))
    .toPandas()
)

payment_type_chart = payment_types.hvplot.bar(
    x="payment_type", y="so_don", width=500, height=330,
    rot=25, title="Số đơn có từng hình thức thanh toán",
    ylabel="Số đơn", xlabel=""
)

installments = (
    payment_lines
    .withColumn("installments_num", F.col("payment_installments").cast("int"))
    .withColumn(
        "installment_group",
        F.when(F.col("installments_num").isNull(), "Thiếu/không đọc được")
        .when(F.col("installments_num") <= 0, "0 hoặc âm")
        .when(F.col("installments_num") == 1, "1")
        .when(F.col("installments_num") <= 3, "2–3")
        .when(F.col("installments_num") <= 6, "4–6")
        .when(F.col("installments_num") <= 12, "7–12")
        .otherwise("≥13")
    )
)

installment_counts = installments.groupBy("installment_group").count().toPandas()
installment_counts["thu_tu"] = installment_counts["installment_group"].map(
    {"0 hoặc âm": 0, "1": 1, "2–3": 2, "4–6": 3,
     "7–12": 4, "≥13": 5, "Thiếu/không đọc được": 6}
)
installment_counts = installment_counts.sort_values("thu_tu")
installment_chart = installment_counts.hvplot.bar(
    x="installment_group", y="count", width=500, height=330,
    title="Số kỳ trả góp theo dòng thanh toán",
    ylabel="Số dòng thanh toán", xlabel=""
)

quality = pd.DataFrame([
    {
        "Kiểm tra": "Đơn delivered theo trạng thái đối chiếu tiền",
        "Kết quả": order_values.groupBy("reconciliation_status")
            .count().orderBy("reconciliation_status").toPandas().to_dict("records")
    },
    {
        "Kiểm tra": "Review ID gắn với nhiều order_id",
        "Kết quả": review_ids.filter(F.col("linked_orders") > 1).count()
    },
    {
        "Kiểm tra": "Review ID có điểm không nhất quán",
        "Kết quả": review_ids.filter(F.col("different_scores") > 1).count()
    },
    {
        "Kiểm tra": "Item thiếu danh mục sản phẩm",
        "Kết quả": item_categories.filter(F.col("category") == "Thiếu danh mục").count()
    },
    {
        "Kiểm tra": "Dòng thanh toán có installments bằng 0 hoặc âm",
        "Kết quả": installments.filter(F.col("installment_group") == "0 hoặc âm").count()
    }
])

print("PHÂN VỊ TRÊN TOÀN BỘ GIÁ TRỊ HỢP LỆ")
display(summary)

print("CÁC TRƯỜNG HỢP CẦN TÁCH RIÊNG KHI DIỄN GIẢI")
display(quality)

print("PHƯƠNG THỨC THANH TOÁN: MỘT ĐƠN CÓ THỂ XUẤT HIỆN Ở NHIỀU PHƯƠNG THỨC")
display(payment_types)

display(
    pn.Column(
        pn.Row(payment_chart, freight_chart),
        pn.Row(item_chart, delivery_chart),
        pn.Row(review_chart, late_chart),
        pn.Row(category_chart, payment_type_chart),
        installment_chart
    )
)


PHÂN VỊ TRÊN TOÀN BỘ GIÁ TRỊ HỢP LỆ


,Chỉ tiêu,Số dòng,Có giá trị,Thiếu,Nhỏ nhất,P50,P75,P90,P95,P99,Lớn nhất
0,Tiền thanh toán/đơn khớp,95908,95908,0,9.59,105.07,175.82,304.18,442.59,1043.32,13664.08
1,Phí vận chuyển/đơn khớp,95908,95908,0,0.00,17.16,23.92,39.06,54.24,102.55,1794.96
2,Số item/đơn delivered,96478,96478,0,1.00,1.00,1.00,1.00,2.00,3.00,21.00
3,Số ngày từ đặt đến giao hợp lệ,96478,96281,197,0.53,10.23,15.73,23.10,29.27,46.01,209.63


CÁC TRƯỜNG HỢP CẦN TÁCH RIÊNG KHI DIỄN GIẢI


,Kiểm tra,Kết quả
0,Đơn delivered theo trạng thái đối chiếu tiền,"[{'reconciliation_status': 'KHOP', 'count': 95..."
1,Review ID gắn với nhiều order_id,789
2,Review ID có điểm không nhất quán,0
3,Item thiếu danh mục sản phẩm,1537
4,Dòng thanh toán có installments bằng 0 hoặc âm,2


PHƯƠNG THỨC THANH TOÁN: MỘT ĐƠN CÓ THỂ XUẤT HIỆN Ở NHIỀU PHƯƠNG THỨC


,payment_type,so_dong_thanh_toan,so_don
0,credit_card,74586,74304
1,boleto,19191,19191
2,voucher,5493,3679
3,debit_card,1486,1485


Column
    [0] Row
        [0] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [1] Row
        [0] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [2] Row
        [0] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [3] Row
        [0] HoloViews(Bars, height=460, sizing_mode='fixed', width=650)
        [1] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)
    [4] HoloViews(Bars, height=330, sizing_mode='fixed', width=500)

### Cell code 09: Xem xu hướng tháng và khác biệt thời kỳ

- **Mục đích:** Thống kê đơn, khách, review và giao hàng theo tháng; so phân bố tiền, item, giao hàng giữa các giai đoạn.
- **Vì sao:** Hành vi mua và chất lượng dữ liệu có thể thay đổi theo thời gian.
- **Giúp ích:** Nhận diện biến động trước khi đánh giá mô hình trên giai đoạn tương lai.


In [11]:
import pandas as pd
import hvplot.pandas
import panel as pn

from pyspark.sql import functions as F

pn.extension()

eda = eda_orders

delivered_orders = (
    eda
    .select(
        "order_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "item_count",
        "total_payment",
        "reconciliation_status"
    )
    .dropDuplicates(
        ["order_id"]
    )
)

delivery_features = (
    orders_prepared
    .select(
        "order_id",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "co_the_tinh_dac_trung_giao_hang",
        "so_ngay_tu_dat_den_giao",
        "giao_tre_so_voi_du_kien"
    )
    .dropDuplicates(
        ["order_id"]
    )
)

order_timeline = (
    delivered_orders
    .join(
        delivery_features,
        "order_id",
        "left"
    )
    .withColumn(
        "purchased_at",
        F.to_timestamp(
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "purchase_month",
        F.trunc(
            F.to_date(
                "purchased_at"
            ),
            "month"
        )
    )
    .withColumn(
        "delivery_days",
        F.when(
            F.col(
                "co_the_tinh_dac_trung_giao_hang"
            ) == 1,
            F.col(
                "so_ngay_tu_dat_den_giao"
            ).cast("double")
        )
    )
    .withColumn(
        "late_flag",
        F.when(
            (
                F.col(
                    "co_the_tinh_dac_trung_giao_hang"
                ) == 1
            )
            &
            F.col(
                "giao_tre_so_voi_du_kien"
            ).isNotNull(),
            F.col(
                "giao_tre_so_voi_du_kien"
            ).cast("int")
        )
    )
    .withColumn(
        "matched_payment",
        F.when(
            F.col(
                "reconciliation_status"
            ) == "KHOP",
            F.col(
                "total_payment"
            ).cast("double")
        )
    )
)

timeline_rows = (
    order_timeline.count()
)

timeline_unique = (
    order_timeline
    .select("order_id")
    .distinct()
    .count()
)

if timeline_rows != timeline_unique:
    raise ValueError(
        "order_timeline bị nhân dòng sau join."
    )

date_range = (
    order_timeline
    .agg(
        F.min(
            "purchased_at"
        ).alias(
            "first_purchase"
        ),
        F.max(
            "purchased_at"
        ).alias(
            "last_purchase"
        )
    )
    .first()
)

monthly_orders = (
    order_timeline
    .filter(
        F.col(
            "purchase_month"
        ).isNotNull()
    )
    .groupBy(
        "purchase_month"
    )
    .agg(
        F.count("*").alias(
            "so_don"
        ),
        F.countDistinct(
            "customer_unique_id"
        ).alias(
            "so_khach"
        ),
        F.sum(
            F.when(
                F.col(
                    "reconciliation_status"
                ) == "KHOP",
                1
            ).otherwise(0)
        ).alias(
            "so_don_tien_khop"
        ),
        F.sum(
            F.when(
                F.col(
                    "reconciliation_status"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "so_don_thieu_doi_chieu_tien"
        ),
        F.sum(
            F.when(
                (
                    F.col(
                        "reconciliation_status"
                    ).isNotNull()
                )
                &
                (
                    F.col(
                        "reconciliation_status"
                    ) != "KHOP"
                ),
                1
            ).otherwise(0)
        ).alias(
            "so_don_tien_khong_khop"
        ),
        F.percentile_approx(
            "matched_payment",
            [0.5, 0.9],
            10000
        ).alias(
            "phan_vi_tien"
        ),
        F.percentile_approx(
            "item_count",
            [0.5, 0.9],
            10000
        ).alias(
            "phan_vi_item"
        ),
        F.percentile_approx(
            "delivery_days",
            [0.5, 0.9],
            10000
        ).alias(
            "phan_vi_giao"
        ),
        F.count(
            "delivery_days"
        ).alias(
            "so_don_giao_hop_le"
        ),
        F.count(
            "late_flag"
        ).alias(
            "so_don_du_dieu_kien_xet_giao_tre"
        ),
        F.sum(
            F.coalesce(
                F.col("late_flag"),
                F.lit(0)
            )
        ).alias(
            "so_don_giao_tre"
        )
    )
    .withColumn(
        "ty_le_tien_khop_pct",
        F.round(
            100
            * F.col(
                "so_don_tien_khop"
            )
            / F.col("so_don"),
            2
        )
    )
    .withColumn(
        "ty_le_giao_tre_pct",
        F.when(
            F.col(
                "so_don_du_dieu_kien_xet_giao_tre"
            ) > 0,
            F.round(
                100
                * F.col(
                    "so_don_giao_tre"
                )
                / F.col(
                    "so_don_du_dieu_kien_xet_giao_tre"
                ),
                2
            )
        )
    )
    .select(
        "purchase_month",
        "so_don",
        "so_khach",
        "so_don_tien_khop",
        "so_don_tien_khong_khop",
        "so_don_thieu_doi_chieu_tien",
        "ty_le_tien_khop_pct",
        F.round(
            F.col(
                "phan_vi_tien"
            )[0],
            2
        ).alias(
            "tien_p50"
        ),
        F.round(
            F.col(
                "phan_vi_tien"
            )[1],
            2
        ).alias(
            "tien_p90"
        ),
        F.col(
            "phan_vi_item"
        )[0].alias(
            "item_p50"
        ),
        F.col(
            "phan_vi_item"
        )[1].alias(
            "item_p90"
        ),
        "so_don_giao_hop_le",
        F.round(
            F.col(
                "phan_vi_giao"
            )[0],
            2
        ).alias(
            "ngay_giao_p50"
        ),
        F.round(
            F.col(
                "phan_vi_giao"
            )[1],
            2
        ).alias(
            "ngay_giao_p90"
        ),
        "so_don_du_dieu_kien_xet_giao_tre",
        "so_don_giao_tre",
        "ty_le_giao_tre_pct"
    )
    .orderBy(
        "purchase_month"
    )
)

review_events = (
    review_links_processed
    .filter(
        F.col(
            "review_id"
        ).isNotNull()
    )
    .groupBy(
        "review_id"
    )
    .agg(
        F.min(
            F.to_date(
                "review_creation_date"
            )
        ).alias(
            "review_created_on"
        ),
        F.countDistinct(
            "order_id"
        ).alias(
            "linked_orders"
        )
    )
    .withColumn(
        "review_month",
        F.trunc(
            F.col(
                "review_created_on"
            ),
            "month"
        )
    )
)

monthly_reviews = (
    review_events
    .filter(
        F.col(
            "review_month"
        ).isNotNull()
    )
    .groupBy(
        "review_month"
    )
    .agg(
        F.count("*").alias(
            "so_review_id"
        ),
        F.sum(
            F.when(
                F.col(
                    "linked_orders"
                ) > 1,
                1
            ).otherwise(0)
        ).alias(
            "so_review_id_nhieu_don"
        )
    )
    .orderBy(
        "review_month"
    )
)

monthly_orders_pd = (
    monthly_orders
    .toPandas()
)

monthly_reviews_pd = (
    monthly_reviews
    .toPandas()
)

monthly_orders_pd[
    "purchase_month"
] = pd.to_datetime(
    monthly_orders_pd[
        "purchase_month"
    ]
)

monthly_reviews_pd[
    "review_month"
] = pd.to_datetime(
    monthly_reviews_pd[
        "review_month"
    ]
)

print("=" * 70)
print("PHẠM VI THỜI GIAN CỦA DỮ LIỆU EDA")
print("=" * 70)

print(
    "Ngày mua đầu tiên:",
    date_range[
        "first_purchase"
    ]
)

print(
    "Ngày mua cuối cùng:",
    date_range[
        "last_purchase"
    ]
)

print(
    "\nKhông chia Train/Test trong EDA."
)

print(
    "Temporal split và prediction window "
    "sẽ được thực hiện ở bước Churn Labeling."
)

print(
    "\nXU HƯỚNG DELIVERED ORDERS "
    "THEO THÁNG"
)

display(
    monthly_orders_pd
)

print(
    "\nREVIEW THEO THÁNG TẠO "
    "- MỖI REVIEW_ID ĐẾM MỘT LẦN"
)

display(
    monthly_reviews_pd
)

orders_chart = (
    monthly_orders_pd
    .hvplot.line(
        x="purchase_month",
        y="so_don",
        width=620,
        height=330,
        title=(
            "Số đơn delivered "
            "theo tháng"
        ),
        xlabel="Tháng mua",
        ylabel="Số đơn",
        marker="circle"
    )
)

customers_chart = (
    monthly_orders_pd
    .hvplot.line(
        x="purchase_month",
        y="so_khach",
        width=620,
        height=330,
        title=(
            "Số khách hàng "
            "theo tháng"
        ),
        xlabel="Tháng mua",
        ylabel="Số khách",
        marker="circle"
    )
)

reviews_chart = (
    monthly_reviews_pd
    .hvplot.line(
        x="review_month",
        y="so_review_id",
        width=620,
        height=330,
        title=(
            "Số review_id "
            "theo tháng tạo review"
        ),
        xlabel="Tháng tạo review",
        ylabel="Số review_id",
        marker="circle"
    )
)

money_chart = (
    monthly_orders_pd
    .hvplot.line(
        x="purchase_month",
        y=[
            "tien_p50",
            "tien_p90"
        ],
        width=620,
        height=330,
        title=(
            "P50 và P90 giá trị thanh toán "
            "của đơn đối chiếu khớp"
        ),
        xlabel="Tháng mua",
        ylabel="Giá trị thanh toán"
    )
)

item_chart_month = (
    monthly_orders_pd
    .hvplot.line(
        x="purchase_month",
        y=[
            "item_p50",
            "item_p90"
        ],
        width=620,
        height=330,
        title=(
            "P50 và P90 số item/đơn "
            "theo tháng"
        ),
        xlabel="Tháng mua",
        ylabel="Số item"
    )
)

delivery_chart_month = (
    monthly_orders_pd
    .hvplot.line(
        x="purchase_month",
        y=[
            "ngay_giao_p50",
            "ngay_giao_p90"
        ],
        width=620,
        height=330,
        title=(
            "P50 và P90 thời gian giao hàng "
            "hợp lệ theo tháng"
        ),
        xlabel="Tháng mua",
        ylabel="Số ngày"
    )
)

late_chart_month = (
    monthly_orders_pd
    .hvplot.line(
        x="purchase_month",
        y="ty_le_giao_tre_pct",
        width=620,
        height=330,
        title=(
            "Tỷ lệ giao trễ "
            "theo tháng mua"
        ),
        xlabel="Tháng mua",
        ylabel="Tỷ lệ (%)",
        marker="circle"
    )
)

payment_quality_chart = (
    monthly_orders_pd
    .hvplot.line(
        x="purchase_month",
        y="ty_le_tien_khop_pct",
        width=620,
        height=330,
        title=(
            "Tỷ lệ đơn đối chiếu tiền khớp "
            "theo tháng"
        ),
        xlabel="Tháng mua",
        ylabel="Tỷ lệ (%)",
        marker="circle"
    )
)

display(
    pn.Column(
        pn.Row(
            orders_chart,
            customers_chart
        ),
        pn.Row(
            reviews_chart,
            money_chart
        ),
        pn.Row(
            item_chart_month,
            delivery_chart_month
        ),
        pn.Row(
            late_chart_month,
            payment_quality_chart
        )
    )
)

PHẠM VI THỜI GIAN CỦA DỮ LIỆU EDA
Ngày mua đầu tiên: 2016-09-15 12:16:38
Ngày mua cuối cùng: 2018-08-29 15:00:37

Không chia Train/Test trong EDA.
Temporal split và prediction window sẽ được thực hiện ở bước Churn Labeling.

XU HƯỚNG DELIVERED ORDERS THEO THÁNG


,purchase_month,so_don,so_khach,so_don_tien_khop,so_don_tien_khong_khop,so_don_thieu_doi_chieu_tien,ty_le_tien_khop_pct,tien_p50,tien_p90,item_p50,item_p90,so_don_giao_hop_le,ngay_giao_p50,ngay_giao_p90,so_don_du_dieu_kien_xet_giao_tre,so_don_giao_tre,ty_le_giao_tre_pct
0,2016-09-01,1,1,0,1,0,0.00,NaN,NaN,3,3,1,54.81,54.81,1,1,100.00
1,2016-10-01,265,262,260,5,0,98.11,111.22,383.24,1,2,261,17.96,34.96,261,2,0.77
2,2016-12-01,1,1,1,0,0,100.00,19.62,19.62,1,1,1,4.69,4.69,1,0,0.00
3,2017-01-01,750,718,740,10,0,98.67,103.46,374.28,1,2,747,10.71,19.27,747,22,2.95
4,2017-02-01,1653,1630,1640,13,0,99.21,106.32,317.20,1,1,1638,10.95,22.27,1638,49,2.99
5,2017-03-01,2546,2508,2537,9,0,99.65,101.28,317.16,1,2,2545,10.13,21.15,2545,116,4.56
6,2017-04-01,2303,2274,2289,14,0,99.39,103.55,319.44,1,1,2279,12.75,25.46,2279,150,6.58
7,2017-05-01,3546,3479,3521,25,0,99.29,104.37,298.84,1,1,3530,9.79,19.30,3530,106,3.00
8,2017-06-01,3135,3076,3118,17,0,99.46,100.33,291.37,1,1,3130,10.80,19.79,3130,94,3.00
9,2017-07-01,3872,3802,3850,22,0,99.43,102.00,267.02,1,2,3841,10.17,19.79,3841,108,2.81



REVIEW THEO THÁNG TẠO - MỖI REVIEW_ID ĐẾM MỘT LẦN


,review_month,so_review_id,so_review_id_nhieu_don
0,2016-10-01,176,3
1,2016-11-01,101,0
2,2016-12-01,45,0
3,2017-01-01,232,7
4,2017-02-01,1392,19
5,2017-03-01,2459,22
6,2017-04-01,2036,18
7,2017-05-01,3670,39
8,2017-06-01,3393,47
9,2017-07-01,3460,43


Column
    [0] Row
        [0] HoloViews(Curve, height=330, sizing_mode='fixed', width=620)
        [1] HoloViews(Curve, height=330, sizing_mode='fixed', width=620)
    [1] Row
        [0] HoloViews(Curve, height=330, sizing_mode='fixed', width=620)
        [1] HoloViews(NdOverlay, height=330, sizing_mode='fixed', width=620)
    [2] Row
        [0] HoloViews(NdOverlay, height=330, sizing_mode='fixed', width=620)
        [1] HoloViews(NdOverlay, height=330, sizing_mode='fixed', width=620)
    [3] Row
        [0] HoloViews(Curve, height=330, sizing_mode='fixed', width=620)
        [1] HoloViews(Curve, height=330, sizing_mode='fixed', width=620)

## Kết luận EDA

EDA sử dụng trực tiếp bảng nền `olist_eda_delivered_orders` được tạo từ bước Processing, với grain **1 dòng = 1 delivered order**. Pipeline phân tích đi từ **tổng quan hành vi mua → one-time/repeat customer → cohort → inter-purchase time → phân bố các đặc trưng giao dịch/chất lượng → xu hướng theo thời gian**.

Các output chính của notebook gồm: quy mô và tỷ lệ khách mua một lần/mua lại; tỷ lệ mua lần hai theo cohort ở các cửa sổ 30/60/90/120/180 ngày; phân bố và các phân vị P50/P75/P90/P95 của khoảng cách giữa các lần mua; phân bố payment, freight, item, delivery, review và category; cùng các xu hướng theo tháng và khác biệt giữa các giai đoạn dữ liệu.

Các kết quả này tạo cơ sở mô tả hành vi mua lại và độ dài chu kỳ mua của khách hàng trước khi chuyển sang **Churn Labeling**. Ở bước tiếp theo, các ngưỡng 90/120/180 ngày sẽ được xây dựng thành nhãn theo prediction date và kiểm soát right censoring/leakage; EDA chỉ cung cấp bằng chứng thực nghiệm để khảo sát các ngưỡng, không tạo target churn chính thức.
